# 8.1 · Caching and eviction

*Systems and scale* · 8 exercises

Keep recent answers close: LRU and other eviction policies, and how to measure a hit rate.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-8.1) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-8.1) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-8.1)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import random
from collections import OrderedDict, deque

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 8.1.1 · LRU cache from the standard library  `Core`

**Task.** Write `LRUCache(capacity)` with `get(key)` (returns `None` on a miss) and `put(key, value)`, evicting the least recently used key. Use `move_to_end` and `popitem(last=False)`.

**Example.** capacity 2: put a, put b, get a, put c -> b is evicted; get b is `None`

**Big O.** O(1) per operation.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-1)

In [ ]:
# 8.1.1 LRU cache from the standard library
class LRUCache:
    def __init__(self, capacity):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

In [ ]:
# Test helpers: run once.
def check_lru(cls):
    c = cls(2)
    c.put("a", 1)
    c.put("b", 2)
    assert c.get("a") == 1
    c.put("c", 3)
    assert c.get("b") is None and c.get("a") == 1 and c.get("c") == 3
    c.put("a", 10)
    assert c.get("a") == 10

In [ ]:
def test_ex_8_1_1():
    check_lru(LRUCache)


test_ex_8_1_1()
print('8.1.1 passed')

### 8.1.2 · LRU from a hash table and a doubly linked list  `Core`

**Task.** Build `LinkedLRU(capacity)` without OrderedDict: a dict from key to node, plus a doubly linked list with head and tail sentinels. Most recent sits next to head; evict from next to tail.

**Example.** Same behaviour as 2.4.1

**Big O.** O(1) per get and put: the dict finds the node, the links let it move without shifting.

**Note.** Sentinel nodes remove every "is the list empty?" special case from insert and unlink.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-2)

In [ ]:
# 8.1.2 LRU from a hash table and a doubly linked list
class _Node:
    __slots__ = ("key", "value", "prev", "next")

    def __init__(self, key=None, value=None):
        raise NotImplementedError


class LinkedLRU:
    def __init__(self, capacity):
        raise NotImplementedError

    def _unlink(self, node):
        raise NotImplementedError

    def _push_front(self, node):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

In [ ]:
def test_ex_8_1_2():
    check_lru(LinkedLRU)
    rng = random.Random(4)
    ref, test = (LRUCache(5), LinkedLRU(5))
    for _ in range(2000):
        k = rng.randrange(12)
        if rng.random() < 0.5:
            assert ref.get(k) == test.get(k)
        else:
            v = rng.random()
            ref.put(k, v)
            test.put(k, v)


test_ex_8_1_2()
print('8.1.2 passed')

### 8.1.3 · Measure a hit rate  `Core`

**Task.** Write `hit_rate(cache, requests)` that, for each requested key, counts a hit when `get` finds it and otherwise `put`s it. Return hits / requests.

**Example.** `hit_rate(LRUCache(2), ["a", "b", "a", "a", "c", "b"])` -> `0.333...` (2 hits in 6)

**Big O.** O(R) for R requests.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-3)

In [ ]:
# 8.1.3 Measure a hit rate
def hit_rate(cache, requests):
    raise NotImplementedError

In [ ]:
def test_ex_8_1_3():
    assert abs(hit_rate(LRUCache(2), ['a', 'b', 'a', 'a', 'c', 'b']) - 2 / 6) < 1e-09


test_ex_8_1_3()
print('8.1.3 passed')

### 8.1.4 · FIFO eviction for comparison  `Core`

**Task.** Write `FIFOCache(capacity)` that evicts the oldest inserted key regardless of use, with the same `get`/`put` interface. Compare its hit rate with LRU on a stream that reuses a few hot keys.

**Example.** hot keys reused constantly -> LRU's hit rate is higher than FIFO's

**Big O.** O(1) per operation with a deque of insertion order.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-4)

In [ ]:
# 8.1.4 FIFO eviction for comparison
class FIFOCache:
    def __init__(self, capacity):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

In [ ]:
def test_ex_8_1_4():
    c = FIFOCache(2)
    c.put('a', 1)
    c.put('b', 2)
    c.get('a')
    c.put('c', 3)
    assert c.get('a') is None
    assert c.get('b') == 2
    assert c.get('c') == 3
    rng = random.Random(9)
    hot = [rng.choice('ab') if rng.random() < 0.7 else str(rng.randrange(50)) for _ in range(3000)]
    assert hit_rate(LRUCache(5), hot) > hit_rate(FIFOCache(5), hot)


test_ex_8_1_4()
print('8.1.4 passed')

### 8.1.5 · Random eviction in O(1)  `Core`

**Task.** Write `RandomCache(capacity, rng)` that evicts a uniformly random key. Keep keys in a list plus a dict of key to index; remove by swapping the victim with the last slot.

**Example.** capacity 3 never holds more than 3 keys; which key goes depends on the seed

**Big O.** O(1) per operation.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-5)

In [ ]:
# 8.1.5 Random eviction in O(1)
class RandomCache:
    def __init__(self, capacity, rng):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

In [ ]:
def test_ex_8_1_5():
    rc = RandomCache(3, random.Random(1))
    for k in range(20):
        rc.put(k, k)
        assert len(rc.data) <= 3
        assert len(rc.keys) == len(rc.data) == len(rc.pos)
        assert all((rc.keys[rc.pos[x]] == x for x in rc.keys))
    assert all((rc.get(k) == k for k in rc.keys))


test_ex_8_1_5()
print('8.1.5 passed')

### 8.1.6 · LRU's worst case, fixed by randomness  `Practice`

**Task.** Write `cyclic_requests(capacity, rounds)` that cycles through capacity + 1 keys. Show LRU's hit rate on it is 0, while `RandomCache` keeps a healthy share of hits.

**Example.** capacity 10, 100 rounds -> LRU `0.0`; random eviction (seed 1) well above `0.5`

**Big O.** O(R). Every request evicts the very key that will be needed next, the textbook LRU pathology.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-6)

In [ ]:
# 8.1.6 LRU's worst case, fixed by randomness
def cyclic_requests(capacity, rounds):
    raise NotImplementedError

In [ ]:
def test_ex_8_1_6():
    req = cyclic_requests(10, 100)
    assert hit_rate(LRUCache(10), req) == 0.0
    assert hit_rate(RandomCache(10, random.Random(1)), req) > 0.5


test_ex_8_1_6()
print('8.1.6 passed')

### 8.1.7 · A bounded memoizing wrapper  `Practice`

**Task.** Write `lru_memo(maxsize)` returning a decorator that caches results in your `LRUCache`, keyed by the argument tuple, and exposes `wrapper.hits` and `wrapper.misses`.

**Example.** decorate a slow `square(x)`, call it with 1, 2, 1 -> hits `1`, misses `2`

**Big O.** O(1) bookkeeping per call; memory is capped at maxsize entries, unlike an unbounded dict.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-7)

In [ ]:
# 8.1.7 A bounded memoizing wrapper
def lru_memo(maxsize):
    raise NotImplementedError

In [ ]:
def test_ex_8_1_7():

    @lru_memo(2)
    def square(x):
        return x * x
    assert [square(1), square(2), square(1)] == [1, 4, 1]
    assert (square.hits, square.misses) == (1, 2)

    @lru_memo(4)
    def nothing(x):
        return None
    nothing(1)
    nothing(1)
    assert nothing.hits == 1


test_ex_8_1_7()
print('8.1.7 passed')

### 8.1.8 · Cache-friendly traversal order  `Core`

**Task.** Write `sum_rows_first(grid)` and `sum_columns_first(grid)`, then a `blocked_transpose(m, block)` that copies the matrix tile by tile. All must give correct results; time them on a 2000 by 2000 grid.

**Example.** both sums of `[[1, 2], [3, 4]]` -> `10`; `blocked_transpose([[1, 2, 3], [4, 5, 6]], 2)` -> `[[1, 4], [2, 5], [3, 6]]`

**Big O.** All O(R * C) steps. Row-first touches memory in order (spatial locality); in pure Python the gap is smaller than in C, because each list element is a pointer to an object.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-1-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-1-8)

In [ ]:
# 8.1.8 Cache-friendly traversal order
def sum_rows_first(grid):
    raise NotImplementedError


def sum_columns_first(grid):
    raise NotImplementedError


def blocked_transpose(m, block):
    raise NotImplementedError

In [ ]:
def test_ex_8_1_8():
    g = [[1, 2], [3, 4]]
    assert sum_rows_first(g) == 10 == sum_columns_first(g)
    assert blocked_transpose([[1, 2, 3], [4, 5, 6]], 2) == [[1, 4], [2, 5], [3, 6]]
    m = [[r * 7 + c for c in range(7)] for r in range(5)]
    assert blocked_transpose(m, 3) == [list(col) for col in zip(*m)]


test_ex_8_1_8()
print('8.1.8 passed')